In [ ]:
#GNN-Only Model Code

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from torch.utils.data import TensorDataset, DataLoader

print("\n--- Training GNN-Only Baseline Model ---")

# File paths
SAMPLED_TRANSACTION_PATH = '../data/processed/train_transaction_sample.csv'
GNN_EMBEDDINGS_PATH = '../data/processed/gnn_embeddings.csv'

# Verify file existence
if not os.path.exists(SAMPLED_TRANSACTION_PATH):
    raise FileNotFoundError(
        f"File not found: {SAMPLED_TRANSACTION_PATH}\n"
        "Please ensure your EDA script has run and saved the file to this exact path."
    )

if not os.path.exists(GNN_EMBEDDINGS_PATH):
    raise FileNotFoundError(
        f"File not found: {GNN_EMBEDDINGS_PATH}\n"
        "Please ensure your GNN embedding script has run and saved the file to this exact path."
    )

# Load data
X_transaction_df = pd.read_csv(SAMPLED_TRANSACTION_PATH)
gnn_embeddings_df = pd.read_csv(GNN_EMBEDDINGS_PATH)

# Merge datasets
X_df = X_transaction_df.merge(gnn_embeddings_df, on="TransactionID", how='left')
y = X_df['isFraud'].values
X = X_df.iloc[:, -gnn_embeddings_df.shape[1] + 1:].values  # Only GNN embeddings

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


# Define simple neural network model
class GNNOnlyClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)  
        )

    def forward(self, x):
        return self.fc(x)


# Initialize model, loss, and optimizer
input_dim = X_train_tensor.shape[1]
gnn_only_model = GNNOnlyClassifier(input_dim)

# Class weights for imbalance handling
class_weights = torch.tensor(
    [np.sum(y_train == 0), np.sum(y_train == 1)], dtype=torch.float32
)
class_weights = class_weights.max() / class_weights

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(gnn_only_model.parameters(), lr=1e-4)

# Training loop
epochs = 3
for epoch in range(epochs):
    gnn_only_model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = gnn_only_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Evaluation
gnn_only_model.eval()
with torch.no_grad():
    predictions = gnn_only_model(X_test_tensor)
    y_pred_proba = torch.softmax(predictions, dim=1)[:, 1].numpy()
    y_pred_class = np.argmax(predictions.numpy(), axis=1)

accuracy = accuracy_score(y_test, y_pred_class)
precision = precision_score(y_test, y_pred_class, zero_division=0)
recall = recall_score(y_test, y_pred_class, zero_division=0)
f1 = f1_score(y_test, y_pred_class, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n--- GNN-Only Model Evaluation ---")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print(f"ROC AUC:   {roc_auc:.3f}")



--- Training GNN-Only Baseline Model ---

--- GNN-Only Model Evaluation ---
Accuracy:  0.885
Precision: 0.130
Recall:    0.387
F1 Score:  0.195
ROC AUC:   0.696
